In [ ]:
import numpy as np
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

In [ ]:
def get_householder_vector(x):
    v = np.array(x, dtype=float)
    norm_x = np.sqrt(np.sum(v**2))
    if norm_x == 0:
        return np.zeros_like(v)

    sign = 1.0 if v[0] >= 0 else -1.0
    v[0] += sign * norm_x

    norm_v = np.sqrt(np.sum(v**2))
    v /= norm_v
    return v

In [ ]:
def householder_reduction(A):
    m, n = A.shape
    B = A.copy().astype(float)
    P = np.eye(m)
    Q = np.eye(n)

    print("P:\n", P)
    print()
    print("B:\n", B)
    print()
    print("Q:\n", Q)
    print()

    for k in range(m):
        if k < m - 1:
            x = B[k:m, k]
            v = get_householder_vector(x).reshape(-1, 1)
            if np.any(v):
                B[k:m, k:n] -= 2 * v @ (v.T @ B[k:m, k:n])
                P[:, k:m] -= 2 * (P[:, k:m] @ v) @ v.T
        print("P:\n", P)
        print()
        print("B:\n", B)
        print()
        print("Q:\n", Q)
        print()

        if k < n - 1:
            x = B[k, k+1:n]
            v = get_householder_vector(x).reshape(-1, 1)
            if np.any(v):
                B[k:m, k+1:n] -= 2 * (B[k:m, k+1:n] @ v) @ v.T
                Q[:, k+1:n] -= 2 * (Q[:, k+1:n] @ v) @ v.T
        print("P:\n", P)
        print()
        print("B:\n", B)
        print()
        print("Q:\n", Q)
        print()

    return P, B, Q

In [ ]:
def step2_chase_bump_givens(B, Q):
    m, _ = B.shape
    bump_col = m

    for k in range(m - 1, -1, -1):
        x1 = B[k, k]
        x2 = B[k, bump_col]

        if abs(x2) < 1e-12:
            continue

        r = np.sqrt(x1**2 + x2**2)
        c = x1 / r
        s = x2 / r

        col_k = B[:, k].copy()
        col_bump = B[:, bump_col].copy()
        B[:, k] = c * col_k + s * col_bump
        B[:, bump_col] = -s * col_k + c * col_bump

        q_k = Q[:, k].copy()
        q_bump = Q[:, bump_col].copy()
        Q[:, k] = c * q_k + s * q_bump
        Q[:, bump_col] = -s * q_k + c * q_bump

        print("B:\n", B)
        print()
        print("Q:\n", Q)
        print()

    return B, Q


In [ ]:
def bidiagonalize(A):
    m, n = A.shape
    if m >= n:
        raise ValueError("Алгоритм предназначен для матриц m < n")
    print("Хаусхолдер\n")
    P, B, Q = householder_reduction(A)
    print("\n\n\nГивенс")
    B, Q = step2_chase_bump_givens(B, Q)
    return P, B, Q

In [ ]:
m, n = 4, 6
A = np.random.randn(m, n)

P, B, Q = bidiagonalize(A)

print("\n\nИсходная матрица A:")
print(A)

print("\nВерхняя двудиагональная матрица (B|0):")
print(B)

print("\nОшибка реконструкции ||P * B * Q^T - A||:")
print(np.linalg.norm(P @ B @ Q.T - A))

Хаусхолдер

P:
 [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]

B:
 [[ 0.4967 -0.1383  0.6477  1.523  -0.2342 -0.2341]
 [ 1.5792  0.7674 -0.4695  0.5426 -0.4634 -0.4657]
 [ 0.242  -1.9133 -1.7249 -0.5623 -1.0128  0.3142]
 [-0.908  -1.4123  1.4656 -0.2258  0.0675 -1.4247]]

Q:
 [[1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1.]]

P:
 [[-0.2609 -0.8296 -0.1271  0.477 ]
 [-0.8296  0.4542 -0.0836  0.3138]
 [-0.1271 -0.0836  0.9872  0.0481]
 [ 0.477   0.3138  0.0481  0.8196]]

B:
 [[-1.9036 -1.0311  1.1388 -0.8837  0.6065 -0.2721]
 [ 0.      0.18   -0.1463 -1.0409  0.0897 -0.4907]
 [ 0.     -2.0033 -1.6754 -0.8049 -0.9281  0.3104]
 [ 0.     -1.0746  1.2798  0.6847 -0.2505 -1.4104]]

Q:
 [[1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1.]]

P:
 [[-0.2609 -0.8296 -0.1271  0.477 ]
 [-0.8296  0.4542 -0.0836  0.3138]
 [-0.1271 -0.0836 